# 18j — six-scenario WIS bars by horizon

17j's ranking figure (`res/17j_scenarios_rank_pre_omicron.png`) sorts the series separately in each horizon
panel, so one series sits on a different row in every panel. This notebook draws the **same ranking** with the
rows fixed: horizons 1–4 as columns sharing y ticks.

Run 17j first. The bars and the dot plot read 17j's CSVs only. The age and over-time figures also read the four
forecast caches, because age-level WIS is in no CSV and 95% intervals need the draws. It reads those caches with
`load_9j_assembly`, so it cannot refit. It needs no CoMix or inc2prev read. The scores and coverage are then cached in
`dt_intermediate/18j_age_coverage_*`, so later runs skip the forecast caches and the R pass; set `REBUILD_18J = true` to redo them.

**Rows, top to bottom.** The two no-contact series, then four bars per contact model: contact (frozen) only,
contact (frozen) – variant, contact only, contact – variant.

**Contact-model order.** Each model is scored by its `variant + contact` mean WIS averaged over h1–4. Each level
of each factor (degree: hweibull/negbin; NGM: nbhd/mean/mean-diagonal) is ranked by the best score of any model
with that level, and models are sorted by (degree rank, NGM rank). The degree models therefore stay together, and
a model can sit above one that scores better, so read the scores from the bars and not from the row position.

**Window.** As in 17j's ranking: log-scale WIS over origins up to and including 2021-11-01, before Omicron.

**Second figure: Cleveland dot plot.** The same frame at horizons 1 and 4, as mean WIS relative to no contact – no
variant at that horizon (the line at 1): one row per contact model with its four scenarios, under a no-contact row
with its two (no variant, variant). Both columns share one x-axis. A second version shows all four horizons.

**Age and over-time figures**, at horizons 1 and 4 over the same pre-Omicron origins:
- **Relative WIS by age:** the series' mean WIS for each age group ÷ no contact – no variant's for that age and horizon.
- **WIS and coverage over time:** per-origin mean WIS, and the share of the 7 age groups whose observed infections
  fall inside the 95% interval, with the pandemic periods shaded. Four series are coloured and the rest grey.


In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")
include("9j_viz_utils.jl")
include("15j_viz_utils.jl")
include("17j_viz_utils.jl")
include("18j_viz_utils.jl")
using Statistics, DataFrames, CSV
default_plot_setting()
SAVE_DPI[] = parse(Int, get(ENV, "SAVE_DPI", "600"))   # high-resolution PNG export (save_show; display unchanged)

cfg = FrameworkConfig(constant_contacts = false)   # production defaults: only `horizons` is read here
@assert contacts_label(cfg) == "temporal-w8h-lc0-eng-nuts"
PRE_OMICRON_END = Date(2021, 11, 1)                # = 17j cell 24

In [ ]:
# 17j's ranking, and the check that it is what 17j's per-origin scores give for this window.
rk = CSV.read("../res/17j_scenarios_rank_pre_omicron.csv", DataFrame; stringtype = String)
n_pre = only(unique(rk.n_origins))
@assert nrow(rk) == 22 * length(cfg.horizons) && length(unique(rk.series)) == 22
@assert sort(unique(rk.horizon)) == collect(cfg.horizons) && only(unique(rk.scale)) == WIS_SCALE
@assert n_pre == 55
@assert nrow(unique(rk, [:model, :scenario, :horizon])) == nrow(rk)

let by_dt_h = rename(CSV.read("../res/17j_scenarios_scores_by_model_date_horizon.csv", DataFrame; stringtype = String),
                     :series => :model),
    st = scenario_table(filter(!=(NULL_LABEL), unique(rk.model))),
    re = scenario_rank_frame(scenario_wis_by_horizon_until(by_dt_h, PRE_OMICRON_END), st),
    a = sort(rk, [:series, :horizon]), b = sort(re, [:series, :horizon])
    @assert a.series == b.series && a.horizon == b.horizon && a.rank == b.rank && a.n_origins == b.n_origins
    @assert maximum(abs.(a.wis .- b.wis) ./ a.wis) < 1e-12
end
println("origins ≤ ", PRE_OMICRON_END, ": n = ", n_pre, " | ", length(unique(rk.series)), " series × ",
        length(cfg.horizons), " horizons — reproduced from the per-origin scores")

In [ ]:
# The contact-model order, what it was computed from, and the frame behind the figure.
ord = scenario_bar_contact_order(rk)
show(stdout, MIME("text/plain"), select(ord.models, :model, :mean_wis); allrows = true); println()
show(stdout, MIME("text/plain"), ord.levels; allrows = true); println()

rows = scenario_bar_rows(ord.order)
bar_frame = innerjoin(dropmissing(rows, :scenario), select(rk, :series, :model, :scenario, :horizon, :wis, :rank);
                      on = [:model, :scenario])
@assert nrow(dropmissing(rows, :scenario)) == 22 && nrow(bar_frame) == nrow(rk)   # every series drawn exactly once
sort!(bar_frame, [:horizon, order(:y; rev = true)])
CSV.write("../res/18j_scenarios_bars_by_horizon.csv", bar_frame)
show(stdout, MIME("text/plain"),
     unstack(select(bar_frame, :tick, :model, :y, :horizon, :wis), :horizon, :wis; renamecols = h -> "h$(h)");
     allrows = true, truncate = 40); println()

In [ ]:
save_show(plot_scenario_bars_by_horizon(rk, cfg; title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_scenarios_bars_by_horizon.png");

In [ ]:
# Cleveland dot plot of all 22 series at h1 and h4: mean WIS relative to no contact – no variant at that horizon,
# one row per contact model, under the no-contact row.
DOT_HORIZONS = (1, 4)
let d = scenario_dots_frame(rk; horizons = DOT_HORIZONS)
    @assert nrow(d) == 22 * length(DOT_HORIZONS)
    @assert all(==(1), @subset(d, :scenario .== "no contact").rel_wis)
    show(stdout, MIME("text/plain"), select(unstack(select(d, :model, :horizon, :scenario, :rel_wis), :scenario, :rel_wis),
                                            :model, :horizon, first.(BAR_NULL_ROWS)..., first.(BAR_CONTACT_ROWS)...);
         allrows = true, truncate = 40); println()
end
save_show(plot_scenario_dots(rk; horizons = DOT_HORIZONS, title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_scenarios_dots_h1_h4.png");

In [ ]:
# The same dot plot at all four horizons.
save_show(plot_scenario_dots(rk; horizons = (1, 2, 3, 4), panel_width = 380,
                             title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_scenarios_dots_h1_h2_h3_h4.png");

In [ ]:
# The age-level scores and coverage: from 18j's own cache, or built once from the four forecast caches. Those are
# read with `load_9j_assembly` and never `assemble_or_load_forecasts`, which fits on a miss.
REBUILD_18J = false
grid = cis_age_grid()
combos = vcat([(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                        for nb in (MeanNGM(), NeighbourhoodDegreeNGM())],
              [(NegBinAgePair(),   DiagonalMeanNGM()),      # no-interaction
               (NoContactDegree(), NullNGM())])             # null, in 9j's order
_label(dm, nb) = string(degree_label(dm), "|", ngm_label(nb))
labels_c = [_label(dm, nb) for (dm, nb) in combos]
labels_v = [_label(dm, VariantNGM(nb)) for (dm, nb) in combos]
sources = [(joinpath("../dt_intermediate", "9j_assembly_$(contacts_label(cfg)).jld2"), labels_c, false),
           (frozen_cache_path(cfg), labels_c, true),
           (variant_cache_path(cfg; frozen = false), labels_v, false),
           (variant_cache_path(cfg; frozen = true), [_label(dm, VariantNGM(nb)) for (dm, nb) in combos if needs_stage1(dm)], true)]
scores_pre = scenario_age_coverage_or_load(cfg, sources, unique(rk.series), grid; horizons = DOT_HORIZONS,
                                           origin_max = PRE_OMICRON_END, rebuild = REBUILD_18J)
cov95, cov90 = scores_pre.cov95, scores_pre.cov90

In [ ]:
# Check the scores and the draw-based coverage against 17j's own per-origin scores, on the cached path too.
by_dt_h_csv = rename(CSV.read("../res/17j_scenarios_scores_by_model_date_horizon.csv", DataFrame; stringtype = String),
                     :series => :model)
let got = @subset(scores_pre.by_model_dt_h, :scale .== WIS_SCALE),
    want = @subset(by_dt_h_csv, :scale .== WIS_SCALE, :forecast_date .<= PRE_OMICRON_END, in.(:horizon, Ref(DOT_HORIZONS))),
    j = innerjoin(got, select(want, :model, :forecast_date, :horizon, :wis => :wis_csv); on = [:model, :forecast_date, :horizon]),
    c = innerjoin(select(got, :model => :series, :forecast_date, :horizon, :interval_coverage_90), cov90;
                  on = [:series, :forecast_date, :horizon])
    @assert nrow(got) == nrow(want) == nrow(j) == 22 * n_pre * length(DOT_HORIZONS)
    @assert maximum(abs.(j.wis .- j.wis_csv) ./ j.wis_csv) < 1e-9 "restricted scoring ≠ 17j's per-origin WIS"
    # The tripwire for a transposed age or horizon index: the draw-based coverage at 90% = scoringutils' own.
    @assert nrow(c) == nrow(got) && maximum(abs.(c.interval_coverage_90 .- c.coverage)) < 1e-12 "draw-based 90% coverage ≠ scoringutils"
end
println("checked: WIS against 17j's CSV and draw-based 90% coverage against scoringutils, over ",
        22 * n_pre * length(DOT_HORIZONS), " series × origin × horizon")

In [ ]:
# Relative WIS by age: y = age group, x = the 22 series in the order of the bars, one row per horizon.
age_rel = scenario_age_rel_wis(scores_pre.by_model_h_age)
@assert nrow(age_rel) == 22 * grid.N * length(DOT_HORIZONS) && Set(age_rel.age_group) == Set(grid.LAB)
@assert all(==(1), @subset(age_rel, :series .== NULL_LABEL).rel_wis)
CSV.write("../res/18j_age_rel_wis_h1_h4.csv", sort(age_rel, [:horizon, :series, :age_group]))
println("relative WIS: ", join(["h$(h) $(join(round.(extrema(@subset(age_rel, :horizon .== h).rel_wis); digits = 2), "–"))" for h in DOT_HORIZONS], ", "))
# Colours capped at 1 ± 0.5: mean-diagonal's values reach 2.24 and would wash out the rest. Every cell prints its value.
save_show(plot_scenario_age_heatmap(age_rel, rk, grid; horizons = DOT_HORIZONS, cap = 0.5,
                                    title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_age_rel_wis_h1_h4.png");

In [ ]:
# Mean WIS (left) and 95% coverage (right) over time, with the pandemic periods shaded, one row per horizon.
@assert nrow(cov95) == 22 * n_pre * length(DOT_HORIZONS) && length(unique(cov95.series)) == 22
CSV.write("../res/18j_coverage95_by_origin_h1_h4.csv", cov95)
println("mean 95% coverage over the window: ", join(["$(s) h$(h) $(round(mean(@subset(cov95, :series .== s, :horizon .== h).coverage); digits = 3))"
                                                     for (s, _, _) in OVER_TIME_HIGHLIGHT for h in DOT_HORIZONS], ", "))
wis_time = @subset(rename(by_dt_h_csv, :model => :series), :forecast_date .<= PRE_OMICRON_END)
save_show(plot_scenario_wis_coverage_over_time(wis_time, cov95, grid; horizons = DOT_HORIZONS, nominal = 0.95,
                                               title_suffix = ", origins ≤ $(PRE_OMICRON_END) (n = $(n_pre))"),
          "../res/18j_wis_coverage_over_time_h1_h4.png");